[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bsheese/cs377/blob/main/supplemental/Telco_Pair.ipynb)

# Telco Churn Capstone: Who Gets a Retention Offer?
**CS 377 · companion to 18_1_9_3 Telco Customer Churn · pair activity**

**Driver:** this notebook is yours. Your partner, the **Recorder**, has the **Telco Churn Capstone quiz in Canvas** open, reads each question aloud and types your agreed answers.

**How the two fit together.** The quiz is the guide; this notebook is where you do the work for the quiz questions tagged **💻 NOTEBOOK**. Each section below is named after its quiz question (**Q3** here is **Q3** in the quiz). Questions tagged **💬 NO NOTEBOOK** have no section here: just talk them through.

**For each notebook question:** the code is written for you. The Recorder reads the question aloud; you find the matching section, make the small change marked **✏️** and run the cell (Shift+Enter). Read the result aloud and agree with your partner on what it means. Hints are under each section: try without them first, and open Hint 1 before Hint 2.

## Setup (run this first)
**Part 1 of the quiz (Q1-Q2) has no notebook work.** Run this cell while the Recorder reads Part 1.

It does Sections 1-4 of 18_1_9_3 for you, the way that notebook's TASK comments describe them: it drops `customerID`, makes `TotalCharges` numeric, builds the preprocessing pipeline, makes the 80/20 split and trains XGBoost **with** `scale_pos_weight`. It makes what every section uses:
- `X_train`, `X_test`, `y_train`, `y_test`: 5,634 training and 1,409 test customers; in `y`, 1 means the customer churned and 0 means they stayed.
- `y_proba`: the model's predicted probability of churn for each test customer.
- `oof_proba`: out-of-fold probabilities for the training customers (18_1_9_3 Section 4.3).
- `make_model(weight)`: the same pipeline with `scale_pos_weight=weight`, not yet trained. `SPW` is the weight 18_1_9_3 uses, about 2.77.
- `COST_FN = 200` (a churner who gets no offer) and `COST_FP = 25` (an offer to a customer who would have stayed).
- `summary(y_true, proba, threshold)`: recall, precision, offers sent, missed churners, total cost and accuracy at a threshold.
- `cost_optimal_threshold(y_true, proba)`: the 18_1_9_3 Section 7.2 search over 200 thresholds from 0.01 to 0.99.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import xgboost as xgb
from sklearn.model_selection import train_test_split, cross_val_predict, cross_val_score, GridSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import confusion_matrix, roc_curve, roc_auc_score, average_precision_score

pd.set_option('display.width', 140)

COST_FN = 200.0   # value lost from an unretained churner
COST_FP = 25.0    # cost of a retention offer sent to a non-churner

# Sections 1-3 of 18_1_9_3: load, clean, encode the target, preprocess, split
df = pd.read_csv('https://raw.githubusercontent.com/bsheese/cs377/main/data/Telco-Customer-Churn.csv')
df = df.drop(columns=['customerID'])
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce').fillna(0)
y = (df['Churn'] == 'Yes').astype(int)        # 1 = churned, 0 = stayed
X = df.drop(columns=['Churn'])

numeric_features = X.select_dtypes(include=[np.number]).columns.tolist()
categorical_features = X.select_dtypes(exclude=[np.number]).columns.tolist()
preprocessor = ColumnTransformer(transformers=[
    ('num', Pipeline([('impute', SimpleImputer(strategy='median')), ('scale', StandardScaler())]), numeric_features),
    ('cat', Pipeline([('impute', SimpleImputer(strategy='most_frequent')),
                      ('onehot', OneHotEncoder(handle_unknown='ignore', drop='first'))]), categorical_features),
])
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

SPW = (y_train == 0).sum() / (y_train == 1).sum()   # 4,139 / 1,495, about 2.77


def make_model(weight):
    """The 18_1_9_3 pipeline: preprocessing, then XGBoost with scale_pos_weight=weight (18_1_9_3 uses SPW)."""
    return Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('classifier', xgb.XGBClassifier(scale_pos_weight=weight, n_estimators=100, max_depth=4,
                                         learning_rate=0.1, eval_metric='logloss', random_state=42)),
    ])


def summary(y_true, proba, threshold):
    """Recall, precision, offers sent, missed churners, total cost and accuracy when everyone at or above the
    threshold gets an offer."""
    y_true = np.asarray(y_true)
    pred = np.asarray(proba) >= threshold
    tp = int((pred & (y_true == 1)).sum())
    fp = int((pred & (y_true == 0)).sum())
    fn = int((~pred & (y_true == 1)).sum())
    tn = int((~pred & (y_true == 0)).sum())
    return {'recall': tp / (tp + fn), 'precision': tp / (tp + fp) if tp + fp else 0.0, 'offers sent': tp + fp,
            'missed churners': fn, 'cost': fn * COST_FN + fp * COST_FP, 'accuracy': (tp + tn) / len(y_true)}


def cost_optimal_threshold(y_true, proba):
    """Section 7.2 of 18_1_9_3: try 200 thresholds from 0.01 to 0.99 and keep the cheapest."""
    candidates = np.linspace(0.01, 0.99, 200)
    costs = [summary(y_true, proba, t)['cost'] for t in candidates]
    return candidates[np.argmin(costs)]


# Section 4 of 18_1_9_3: the model and its out-of-fold probabilities
model = make_model(SPW).fit(X_train, y_train)
y_proba = model.predict_proba(X_test)[:, 1]     # P(churn) for the 1,409 test customers
oof_proba = cross_val_predict(make_model(SPW), X_train, y_train, cv=5,
                              method='predict_proba')[:, 1]   # out-of-fold P(churn) for the 5,634 training customers

print(f"training customers: {len(y_train):,} ({y_train.sum()} churned)   "
      f"test customers: {len(y_test):,} ({y_test.sum()} churned)")

# Part 2 (Q3-Q6): Sections 5-7 of 18_1_9_3

## Q3 · The model against no model
> **Quiz Q3:** 18_1_9_3 Section 5.1 compares the model at the default 0.5 threshold with the two strategies from Q1(a). The Q3 cell has everything except the model's own cost: replace `np.nan` with the cost from `fn`, `fp`, `COST_FN` and `COST_FP`, and run it. Does the model beat both strategies, and by how much? Its accuracy is only a little above the 73.5% of calling no one: does that mean it is only a little better?

**✏️ Your change:** Replace `np.nan` with `fn * COST_FN + fp * COST_FP`.

**Tell your Recorder:** the model's cost and how much it saves against each strategy, then what its accuracy does and does not tell you.

<details><summary>Hint 1</summary>

Each missed churner (`fn`) costs `COST_FN`, and each unnecessary offer (`fp`) costs `COST_FP`.

</details>

<details><summary>Hint 2</summary>

`cost_model = fn * COST_FN + fp * COST_FP`

</details>

In [ ]:
y_pred = (y_proba >= 0.5).astype(int)
tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()

cost_call_none = (y_test == 1).sum() * COST_FN
cost_call_all = (y_test == 0).sum() * COST_FP
cost_model = np.nan      # ✏️ the model's cost at 0.5, from fn, fp, COST_FN and COST_FP

print(f'TP {tp}  FN {fn}  FP {fp}  TN {tn}')
print(f'call no one:    ${cost_call_none:,.0f}')
print(f'call everyone:  ${cost_call_all:,.0f}')
print(f'model at 0.5:   ${cost_model:,.0f}')
print(f'model accuracy {(tp + tn) / len(y_test):.3f}   call-no-one accuracy {(y_test == 0).mean():.3f}   '
      f'model recall {tp / (tp + fn):.3f}')

## Q4 · ROC and precision when churn is common
> **Quiz Q4:** This is the Rare Fraud Q3 cell, run on the Telco test set. In fraud, going from 80% to 90% recall raised the FPR by only about 4 points while adding about 3,760 false alarms, and precision fell from 0.81 to 0.03. Before running, say what you expect for Telco, where there are only about 2.8 stayers per churner. Then change `TARGETS` to `[0.80, 0.85, 0.90, 0.95]` and run it. From 80% to 90% recall, how much do the FPR, the false alarms and the precision change? Use the number of customers who stayed (1,035 here, against 85,295 legitimate transactions in fraud) to explain why the ROC curve tells a much more honest story here than it did for fraud (18_1_9_3 discussion question 4).

**✏️ Your change:** Change `TARGETS = [0.80]` to `TARGETS = [0.80, 0.85, 0.90, 0.95]`, then run the cell.

**Tell your Recorder:** your prediction, the change in FPR, false alarms and precision from 80% to 90% recall, then why ROC is more informative here.

<details><summary>Hint 1</summary>

`TARGETS` is a list of recall values. The cell adds one row to the table for each value.

</details>

<details><summary>Hint 2</summary>

`TARGETS = [0.80, 0.85, 0.90, 0.95]`

</details>

In [ ]:
TARGETS = [0.80]      # ✏️ try [0.80, 0.85, 0.90, 0.95]

fpr, tpr, roc_thresholds = roc_curve(y_test, y_proba)
n_churn, n_stay = (y_test == 1).sum(), (y_test == 0).sum()
rows = []
for target in TARGETS:
    i = np.argmax(tpr >= target)          # first point on the curve that reaches the target
    tp, fp = round(tpr[i] * n_churn), round(fpr[i] * n_stay)
    rows.append({'recall target': target, 'recall reached': tpr[i], 'threshold': roc_thresholds[i],
                 'FPR': fpr[i], 'false alarms': fp, 'precision': tp / (tp + fp)})

print(f'test ROC AUC: {roc_auc_score(y_test, y_proba):.3f}   (random guessing: 0.500)')
pd.DataFrame(rows).set_index('recall target').round(4)

## Q5 · Three thresholds, and your prediction
> **Quiz Q5:** 18_1_9_3 Section 7 compares three thresholds on the test set: the default 0.5, Youden's J and the cost-optimal threshold, the last two chosen on the out-of-fold training probabilities. (The cost-optimal threshold will not be Q1(b)'s 0.111; extra-credit Q7 explains why.) The Q5 cell builds that table, but the J threshold is missing. Replace `np.nan` with the threshold where `tpr - fpr` is largest (as in Rare Fraud Q5) and run it. Was your Q2(b) prediction right? Which threshold is cheapest, and how much does it save against 0.5? Look at the accuracy column: what happens to accuracy at the cheapest threshold, and why is that not a problem?

**✏️ Your change:** Replace `np.nan` with `roc_thresholds[np.argmax(tpr - fpr)]`.

**Tell your Recorder:** the three thresholds and costs, whether Q2(b) was right, then what happens to accuracy at the cheapest threshold.

<details><summary>Hint 1</summary>

`np.argmax(tpr - fpr)` gives the position of the largest J; use it to look up the threshold in `roc_thresholds`.

</details>

<details><summary>Hint 2</summary>

`j_threshold = roc_thresholds[np.argmax(tpr - fpr)]`

</details>

In [ ]:
fpr, tpr, roc_thresholds = roc_curve(y_train, oof_proba)
j_threshold = np.nan      # ✏️ the threshold where tpr - fpr is largest
cost_threshold = cost_optimal_threshold(y_train, oof_proba)   # chosen on training data only

rows = []
for name, t in [('Default (0.50)', 0.5), ("Youden's J", j_threshold), ('Cost-optimal', cost_threshold)]:
    rows.append({'strategy': name, 'threshold': t, **summary(y_test, y_proba, t)})
pd.DataFrame(rows).set_index('strategy').round(3)

## Q6 · When the team can only make so many calls
> **Quiz Q6:** 18_1_9_3 discussion question 3 asks why the cost-optimal threshold is so far below 0.5, and what operational constraint might push you to a higher threshold even though it costs more. Suppose the retention team only has time to contact 200 of these 1,409 customers. The Q6 cell compares the cost-optimal threshold with contacting the top `CAPACITY` customers by P(churn); right now `CAPACITY` is 812, the number the cost-optimal threshold contacts. Change it to `200` and run it. What threshold does the limit force, and what happens to recall, precision and cost? Look at the accuracy column too: which strategy is more accurate, and is it the one you would want if the team had no limit? Then explain why the team should still use the model's ranking (and not, say, a random 200 customers) when it can only make 200 calls.

**✏️ Your change:** Change `CAPACITY = 812` to `CAPACITY = 200`, then run the cell.

**Tell your Recorder:** the forced threshold and its recall, precision and cost, what the accuracy column shows, then why the ranking still matters.

<details><summary>Hint 1</summary>

`CAPACITY` is the number of customers the team can contact. The cell sets the threshold so that exactly that many customers are at or above it.

</details>

<details><summary>Hint 2</summary>

`CAPACITY = 200`

</details>

In [ ]:
CAPACITY = 812      # ✏️ the team can only contact 200

cost_threshold = cost_optimal_threshold(y_train, oof_proba)
capacity_threshold = np.sort(y_proba)[::-1][CAPACITY - 1]   # P(churn) of the CAPACITY-th highest customer

rows = []
for name, t in [('Cost-optimal (no limit)', cost_threshold), (f'Top {CAPACITY} only', capacity_threshold)]:
    rows.append({'strategy': name, 'threshold': t, **summary(y_test, y_proba, t)})
pd.DataFrame(rows).set_index('strategy').round(3)

# Extra credit (Q7-Q8): The Break-Even Rule and Nested CV
Only if you have finished Q1-Q6.

## ⭐ Q7 · When does the break-even rule work?
> **Quiz Q7:** Q1(b)'s break-even probability (about 0.11) is lower than the cost-optimal threshold you found in Q5 (about 0.26). The Q7 cell trains the model with two weights and shows each one's mean P(churn), its cost-optimal threshold and its cost at that threshold and at 0.111. Right now both weights are `SPW`. Change the second to `1` and run it. Which model's cost-optimal threshold is close to the break-even value? Compare each model's mean P(churn) with the real churn rate (26.5%) to explain why the rule fits one model and not the other. (Rare Fraud Q6 and Picking the Threshold Q4 showed what a weight does to the mean probability. If a model's cost at 0.111 is a little lower than at its own cost-optimal threshold, that is because the cost-optimal threshold was chosen on the training data, not the test set.)

**✏️ Your change:** Change the second weight in the line marked ✏️ from `SPW` to `1`, then run the cell.

**Tell your Recorder:** each model's mean P(churn), cost-optimal threshold and costs, then why the break-even rule fits one of them.

<details><summary>Hint 1</summary>

The weight list has two entries; only the second one changes.

</details>

<details><summary>Hint 2</summary>

`weights = [SPW, 1]`

</details>

In [ ]:
weights = [SPW, SPW]      # ✏️ change the second weight to 1

rows = []
for w in weights:
    p = make_model(w).fit(X_train, y_train).predict_proba(X_test)[:, 1]
    oof = cross_val_predict(make_model(w), X_train, y_train, cv=5, method='predict_proba')[:, 1]
    t = cost_optimal_threshold(y_train, oof)
    rows.append({'scale_pos_weight': round(w, 2), 'mean P(churn)': p.mean(), 'cost-optimal threshold': t,
                 'cost there': summary(y_test, p, t)['cost'],
                 'cost at 0.111': summary(y_test, p, COST_FP / (COST_FN + COST_FP))['cost']})
pd.DataFrame(rows).set_index('scale_pos_weight').round(3)

## ⭐ Q8 · Nested CV
> **Quiz Q8:** 18_1_9_3 Section 8 estimates how well the whole tuning process (a grid search inside each outer fold) would do on new customers. The Q8 cell runs that nested cross-validation with the 18_1_9_3 grid, but one of the three lists in `param_grid` is missing. Fill it in with `[3, 5]` and run it. Compare the five outer-fold AUCs and their mean with the single test-set AUC from Q4. Then answer 18_1_9_3's discussion question 5: if the five scores had instead been 0.71, 0.79, 0.72, 0.80 and 0.73, what would that mean, and how would you explain it to a manager who wants one number?

**✏️ Your change:** Replace the `...` after `'classifier__max_depth':` with `[3, 5]`.

**Tell your Recorder:** the five outer-fold AUCs, their mean, the comparison with Q4's AUC, then your answer about the unstable scores.

<details><summary>Hint 1</summary>

The other two lists in `param_grid` show the form. The `classifier__` prefix sends the setting to the XGBoost step of the pipeline.

</details>

<details><summary>Hint 2</summary>

`'classifier__max_depth': [3, 5],`

</details>

In [ ]:
param_grid = {
    'classifier__n_estimators':  [50, 100],
    'classifier__max_depth':     ...,          # ✏️ [3, 5]
    'classifier__learning_rate': [0.05, 0.1],
}

inner_grid = GridSearchCV(make_model(SPW), param_grid, cv=3, scoring='roc_auc', n_jobs=-1)
outer_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
nested_scores = cross_val_score(inner_grid, X, y, cv=outer_cv, scoring='roc_auc', n_jobs=-1)

print('Nested CV AUC scores:', np.round(nested_scores, 3))
print(f'Mean: {nested_scores.mean():.3f}  +/-  {nested_scores.std():.3f}')